# Classification task with 3 modes and available interpretable/explainable features

| Prediction task is to determine whether a person makes over 50K a year from 14 available features, numerical and categorical

We will use:

- **Logistic Regression (LR)** as a linear baseline
- **Decision Tree (DT)** as a rule-based model
- **Explainable Boosting Machine (EBM)** as an additive interpretable model

These models were chosen because they show different forms of interpretability:
coefficients, rules, and feature shape functions.

## Feature glossary

The Adult dataset is a binary classification dataset where the target variable is `income`.

### Predictor variables

- `age`: age in years
- `workclass`: employment class
- `fnlwgt`: census sample weighting variable
- `education`: education category
- `education_num`: numerical representation of education level
- `marital_status`: marital-status category
- `occupation`: occupation category
- `relationship`: household relationship category
- `race`: race category
- `sex`: sex category
- `capital_gain`: capital gains
- `capital_loss`: capital losses
- `hours_per_week`: hours worked per week
- `native_country`: native-country category

### Target variable

- `income`: binary target indicating whether annual income is `>50K` or `<=50K`

#  Loading

In [ ]:
# Installs
%pip install numpy pandas matplotlib scikit-learn interpret
# this block is not always required, but the list offers requisites when working outside of a structured space where items have been preset in the environment. The InterpretML is rather large and a refresh of the kernel is sensible 
# after running the cell. 

import the Python packages needed for data handling, preprocessing, modelling, evaluation, and visualisation.

In [ ]:
##these form some imports and then specific features we will use in later blocks, they are placed here at the start to keep everything tidy 
import time
from pathlib import Path


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.io as pio

from interpret import preserve, set_visualize_provider, show
from interpret.glassbox import ExplainableBoostingClassifier # for our RBM method
from interpret.provider import InlineProvider
from interpret.utils import inv_link

from sklearn.compose import ColumnTransformer # for pre-processing pipeline before model fit in regression session
from sklearn.impute import SimpleImputer #useful when dealing with missing values, other options are also available, here we needed some simple options for imputation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
) # confusion matrix, accuracy and classification metrics to read out after model fits.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler #scaling and standardising
from sklearn.tree import DecisionTreeClassifier, plot_tree # for the shallow decision tree

In [ ]:
set_visualize_provider(InlineProvider())
pio.renderers.default = "plotly_mimetype"
# at times in the notebook there are some interactive renderings (html) we want to look at and this will improve how they are presented somewhat

Saving outputs

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Outputs"
FIGURE_DIR = OUTPUT_DIR / "Figures"
MODEL_DIR = OUTPUT_DIR / "Models"
PROCESSED_DIR = OUTPUT_DIR / "Processed_data"

for folder_path in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR, PROCESSED_DIR]:
    folder_path.mkdir(parents=True, exist_ok=True)

#Throughout the notebook, where sensible files, images, and iterations will be saved to the output directory for you to inspect and read. This is the structure set up ahead of time
# save functions in the code will perform the task. HTML files can be downloaded and opened in your browser locally, or in Noteable after clicking 'trust html'
# whilst we don't necessarily need to save all images, when applying a use case workflow, you might need this for tracking, checking and reporting purposes.

## Inspect files

In [ ]:
ADULT_TRAIN_FILE = DATA_DIR / "adult.data"
ADULT_TEST_FILE = DATA_DIR / "adult.test"
ADULT_NAMES_FILE = DATA_DIR / "adult.names"


print("\n--- First 5 lines of adult.data ---")
with open(ADULT_TRAIN_FILE, "r", encoding="utf-8") as file:
    for line_number in range(5):
        print(file.readline().rstrip())

print("\n--- First 5 lines of adult.test ---")
with open(ADULT_TEST_FILE, "r", encoding="utf-8") as file:
    for line_number in range(5):
        print(file.readline().rstrip())

print("\n--- First 15 lines of adult.names ---")
with open(ADULT_NAMES_FILE, "r", encoding="utf-8") as file:
    for line_number in range(15):
        print(file.readline().rstrip())

# Preparation

The raw Adult data files do not contain a header row, so we assign column names manually using the official feature order published in the UCI Adult dataset documentation; pandas supports this via read_csv(..., header=None, names=[...])

### Parsing
#### Assign column names using the official dataset schema

The raw data files do not contain a header row, so we assign column names manually when loading the files.

These names come from the official Adult dataset documentation published by the UCI Machine Learning Repository. The names are assigned in the same order as the fields appear in the raw records.

In [ ]:
adult_column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

adult_train_raw = pd.read_csv(
    ADULT_TRAIN_FILE,
    header=None,
    names=adult_column_names,
    na_values="?",
    skipinitialspace=True
)

adult_test_raw = pd.read_csv(
    ADULT_TEST_FILE,
    header=None,
    names=adult_column_names,
    na_values="?",
    skipinitialspace=True,
    skiprows=1
)

print("adult_train_raw shape:", adult_train_raw.shape)
print("adult_test_raw shape:", adult_test_raw.shape)

adult_train_raw.head()

In [ ]:
# Inspect the loaded training data structure
print("Loaded training columns:")
print(adult_train_raw.columns.tolist())

print("\nNumber of loaded columns:", adult_train_raw.shape[1])
print("Number of expected columns:", len(adult_column_names))

print("\nRaw income label counts in training data:")
print(adult_train_raw["income"].value_counts(dropna=False))

print("\nMissing values per column in training data:")
print(adult_train_raw.isna().sum().sort_values(ascending=False))

In [ ]:
# Inspect income labels before standardisation (formatting)
print("Training income labels before standardisation:")
print(adult_train_raw["income"].value_counts(dropna=False))

print("\nTest income labels before standardisation:")
print(adult_test_raw["income"].value_counts(dropna=False))

# Standardise income labels (basic formatting)
adult_train_raw["income"] = adult_train_raw["income"].astype(str).str.strip()

adult_test_raw["income"] = (
    adult_test_raw["income"]
    .astype(str)
    .str.strip()
    .str.replace(".", "", regex=False)
)

# Inspect income labels after standardisation (formatting)
print("\nTraining income labels after standardisation:")
print(adult_train_raw["income"].value_counts(dropna=False))

print("\nTest income labels after standardisation:")
print(adult_test_raw["income"].value_counts(dropna=False))

## Initial inspection of the target variable and missing values

We have now loaded the raw training and test files and checked the target labels. At this stage, the data is still in its raw tabular form, no train-test split has been created within the training data, and no preprocessing has been learned from the data.

Two useful observations appear immediately:

- The target variable, `income`, is imbalanced: the `<=50K` class is more common than the `>50K` class.
- Missing values are concentrated in a small number of categorical variables, especially `occupation`, `workclass`, and `native_country`.

Checked and standardised the target labels so that both the training and test files use the same two classes. This is for formatting.

In [ ]:
adult_train_raw.info()

adult_train_raw.describe(include="all").T # return dataframe and transpose

## Separate predictor variables from the target variable

Now separate the dataset into:

- **predictor variables** (`X`), which the model will use as inputs
- **target labels** (`y`), which the model will try to predict

In machine learning workflows, `X` is usually stored as a two-dimensional table because it contains multiple feature columns, while `y` is usually stored as a one-dimensional vector because it contains one target value per row.

Separately for:

- the original training dataset, which we will later split into training and validation subsets
- the external test dataset provided with the Adult files, which we will keep untouched until later evaluation

In [ ]:
# Separate input features and target labels from the training file
X_full = adult_train_raw.drop(columns="income")
y_full = adult_train_raw["income"]

# Separate input features and target labels from the original test file
X_external_test = adult_test_raw.drop(columns="income")
y_external_test = adult_test_raw["income"]

print("Training-file feature table shape (X_full):", X_full.shape)
print("Training-file target vector shape (y_full):", y_full.shape)

print("\nTest-file feature table shape (X_external_test):", X_external_test.shape)
print("Test-file target vector shape (y_external_test):", y_external_test.shape)

## Split the training data into a training set and a validation set

We now split the original training file into two parts:

- a **training set**, which will be used to fit preprocessing steps and models
- a **validation set**, which will be used to check model performance during development

We use a **stratified split**, which means the proportion of the two income classes is preserved as closely as possible in both subsets.

The external test file remains untouched at this stage. We will not use it for model development.

In [ ]:
# Split the original training file into a training set and a validation set
X_train, X_valid, y_train, y_valid = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

# Check the shapes of the resulting datasets
print("Training feature table shape (X_train):", X_train.shape)
print("Validation feature table shape (X_valid):", X_valid.shape)
print("Training target vector shape (y_train):", y_train.shape)
print("Validation target vector shape (y_valid):", y_valid.shape)

# Check that class proportions are similar after stratified splitting
print("\nTraining-set class proportions:")
print(y_train.value_counts(normalize=True))

print("\nValidation-set class proportions:")
print(y_valid.value_counts(normalize=True))

What we have so far:

- `X_train` and `X_valid` contain predictor variables only
- `y_train` and `y_valid` contain the target labels
- the external test data remains separate and untouched

### Defining feature types

The Adult dataset contains a mixture of numerical and categorical variables. Before building a preprocessing pipeline, we define which features belong to each group.

This matters because different feature types are handled differently during preprocessing:

- numerical features can be scaled if needed
- categorical features usually need encoding before they can be used by many machine learning models

In [ ]:
# Define numerical features
numerical_features = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

# Define categorical features
categorical_features = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country"
]

# Check the feature groups
print("No. of numerical features:", len(numerical_features))
print("No. of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

## Build a preprocessing pipeline



In [13]:
# Preprocessing for numerical features:
# scale numerical variables for Logistic Regression
numerical_preprocessor = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

# Preprocessing for categorical features:
# make missing values explicit, then one-hot encode categories
categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")), #replaced missing with the term as opposed to other options assigning items in these categories - occupation/or home country.
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing steps across feature types
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_preprocessor, numerical_features),
        ("cat", categorical_preprocessor, categorical_features)
    ]
)

# Full Logistic Regression pipeline
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=5000, random_state=42))
    ]
)

## Preprocessing before model fitting

Before fitting Logistic Regression, we prepare the input features in a way that is appropriate for the data we inspected.

The Adult dataset contains:

- **numerical features**, such as `age`, `education_num`, and `hours_per_week`
- **categorical features**, such as `workclass`, `occupation`, and `native_country`

These feature types need different preprocessing steps.

#### Scaling numerical features

For the numerical features, we apply **standardisation** using `StandardScaler`.

For a numerical variable \(x\), the transformed value \(z\) is:

$$
z = \frac{x - \mu}{\sigma}
$$

- $\mu$ is the mean of that feature in the training data
- $\sigma$ is the standard deviation of that feature in the training data

This transformation centres the feature around 0 and rescales it so that it has unit variance.

We use this because Logistic Regression is a linear model whose optimisation behaves better when numerical features are on comparable scales. In this dataset, variables such as `fnlwgt`, `capital_gain`, and `age` are measured on very different ranges, so scaling helps the model fit more reliably and makes coefficient comparisons more meaningful.

#### Handling missing categorical values

From our inspection of the raw data, missing values are concentrated in the categorical features `workclass`, `occupation`, and `native_country`. missingness was made explicit by assigning a constant label such as `"Missing"`.

If a categorical value is missing, we replace it with:

$$
x_{\text{missing}} \rightarrow \text{"Missing"}
$$

#### Encoding categorical variables

After missing categorical values have been made explicit, we apply **one-hot encoding**.

If a feature has categories such as:

$$
\{\text{Private}, \text{State-gov}, \text{Missing}\}
$$

then one-hot encoding converts this single categorical feature into separate binary indicator columns, for example:

$$
\text{workclass\_Private}, \quad \text{workclass\_State-gov}, \quad \text{workclass\_Missing}
$$

Each row then receives values of 0 or 1 for these columns.

We use one-hot encoding because Logistic Regression requires numerical input,  one-hot encoding is needed for many estimators, especially linear models. We also set `handle_unknown="ignore"` so that if a category appears later in validation or test data that was not seen during training, the pipeline can still transform the data safely. 

#### Combining preprocessing steps in a pipeline

We combine the numerical and categorical preprocessing steps using a `ColumnTransformer`, and then place that inside a full `Pipeline` together with the Logistic Regression model.

This is important because preprocessing choices influence the fitted model directly. It also ensures that all preprocessing steps are learned from the training data only and then applied consistently to the validation and test sets.


# Model fitting -> Logistic Regression

We are using a pipeline, the preprocessing steps and the Logistic Regression model are fitted together as one workflow:

1. the preprocessing steps are learned from the training data
2. the transformed training data is used to fit the model
3. the fitted pipeline is then applied to the validation set to generate predictions

We evaluate the model on the validation set using two types of output:

- **accuracy**, which is the proportion of correct predictions
- a **classification report**, which shows precision, recall, F1-score, and support for each class

This gives us an initial view of model performance before we move on to confusion matrices and interpretation of coefficients.

In [ ]:
# Fit the full preprocessing + Logistic Regression pipeline on the training set
logistic_pipeline.fit(X_train, y_train)

# Generate predictions for the validation set
y_valid_pred_logistic = logistic_pipeline.predict(X_valid)

# Report overall accuracy
print("Validation accuracy:")
print(accuracy_score(y_valid, y_valid_pred_logistic))

# Report class-wise performance
print("\nValidation classification report:")
print(classification_report(y_valid, y_valid_pred_logistic))

### Assessing the confusion matrix

In [ ]:
# Save the confusion matrix figure
confusion_matrix_png = FIGURE_DIR / "01_logreg_01_confusion_matrix.png"
confusion_matrix_svg = FIGURE_DIR / "01_logreg_01_confusion_matrix.svg"

plt.figure()
ConfusionMatrixDisplay.from_predictions(
    y_valid,
    y_valid_pred_logistic,
    display_labels=logistic_pipeline.classes_,
    cmap="Blues"
)
plt.title("Confusion Matrix - Logistic Regression Pipeline")
plt.savefig(confusion_matrix_png, dpi=300, bbox_inches="tight")
plt.savefig(confusion_matrix_svg, bbox_inches="tight")
plt.show()

## Interpreting the confusion matrix and classification metrics

The confusion matrix shows how the predicted class labels compare with the true class labels.

For a binary classification problem, the confusion matrix can be written as:

$$
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
$$

where:

- \(TN\) = **true negatives**: cases correctly predicted as the negative class
- \(FP\) = **false positives**: cases incorrectly predicted as the positive class
- \(FN\) = **false negatives**: cases incorrectly predicted as the negative class
- \(TP\) = **true positives**: cases correctly predicted as the positive class

In scikit-learn, the confusion matrix is defined so that the entry in row \(i\), column \(j\) is the number of observations whose **true** class is \(i\) and whose **predicted** class is \(j\). In the binary case, this corresponds to:

- top-left: \(TN\)
- top-right: \(FP\)
- bottom-left: \(FN\)
- bottom-right: \(TP\) ([scikit-learn.org](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html?utm_source=chatgpt.com))

### Accuracy

The accuracy score is the proportion of all predictions that are correct:

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

Accuracy gives a simple overall summary, but it can be misleading when the class distribution is imbalanced. In this dataset, the `<=50K` class is more common than the `>50K` class, so a model can achieve a good accuracy while still performing less well on the less common class.

### Precision

Precision measures how many predicted positive cases were actually positive:

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

A high precision means that when the model predicts the positive class, it is often correct.

### Recall

Recall measures how many true positive cases were successfully identified:

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

A high recall means that the model captures most of the true positive cases.

### F1-score

The F1-score combines precision and recall using the harmonic mean:

$$
F_1 = \frac{2TP}{2TP + FP + FN}
$$

The F1-score is useful when we want a single summary that balances both false positives and false negatives. Scikit-learn documents the F1-score exactly in this form. ([scikit-learn.org](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html?highlight=f1&utm_source=chatgpt.com))

### Support

Support is simply the number of true observations belonging to each class in the evaluation set.

### Overall

The confusion matrix helps us move beyond a single accuracy value and inspect the types of mistakes the model is making.

In this income classification example, the model performs better on the more common `<=50K` class than on the less common `>50K` class. This is visible in the classification report, where recall is higher for `<=50K` than for `>50K`.

This matters for explainable machine learning because model interpretation should always be read alongside model behaviour. Before we interpret coefficients, we should first understand where the model is performing well and where it is making systematic errors.

In [ ]:
# Save the classification report as a table
classification_report_df = pd.DataFrame(
    classification_report(
        y_valid,
        y_valid_pred_logistic,
        output_dict=True
    )
).T

classification_report_file = PROCESSED_DIR / "01_logreg_02_classification_report.csv"
classification_report_df.to_csv(classification_report_file)

# Bringing in some explainability  - coefficient interpretation

Global interpretations -> these are from the fitted model, they don't specify any causal details but are related to the model specified above, with coefficients speaking to fit. A number of things will influence the coefficients and there are many details they cannot tell us about but they offer an interpretable index.

In [ ]:
# Get feature names after preprocessing
feature_names_after_preprocessing = (
    logistic_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Extract fitted coefficients from the Logistic Regression model
logistic_coefficients = logistic_pipeline.named_steps["model"].coef_[0]

# Combine feature names and coefficients into a table
logistic_coefficients_df = pd.DataFrame({
    "feature": feature_names_after_preprocessing,
    "coefficient": logistic_coefficients
}).sort_values("coefficient", ascending=False)

print(logistic_coefficients_df.head(10)) #showing a few to start with
print()
print(logistic_coefficients_df.tail(10)) #showing a few to start with

In [ ]:
# Save the full coefficient table
coefficient_table_file = PROCESSED_DIR / "01_logreg_03_coefficients_full.csv"
logistic_coefficients_df.to_csv(coefficient_table_file, index=False)

## Interpreting Logistic Regression coefficients

Logistic Regression is an **inherently interpretable model** because it provides a direct coefficient for each input feature in the transformed feature space.

For a binary classification problem, the model can be written as:

$$
\log\left(\frac{p(x)}{1-p(x)}\right) = \beta_0 + \sum_{j=1}^{p} \beta_j x_j
$$

where:

- \(p(x)\) is the predicted probability of the positive class
- \(\beta_0\) is the intercept
- \(\beta_j\) is the coefficient for feature \(x_j\)

The left-hand side is called the **log-odds** or **logit**.

### What a coefficient means

Each coefficient tells us how the model changes the **log-odds** of the positive class when that feature increases by one unit, **holding the other transformed features constant**.

- if $\beta_j > 0$, increasing that feature pushes the prediction more toward the positive class
- if $\beta_j < 0$, increasing that feature pushes the prediction more toward the negative class
- if $|\beta_j|$ is larger, that feature has a stronger effect within the fitted model

### Odds ratios

If we exponentiate a coefficient, we obtain an **odds ratio**:

$$
\text{Odds Ratio}_j = e^{\beta_j}
$$

This can be helpful for interpretation:

- if $e^{\beta_j} > 1$, a one-unit increase in that feature is associated with higher odds of the positive class
- if $e^{\beta_j} < 1$, a one-unit increase in that feature is associated with lower odds of the positive class

### Why preprocessing matters

The coefficients must be interpreted in the **processed feature space**

- numerical features were standardised, so numerical coefficients are expressed in standardised units
- categorical features were one-hot encoded, so each category has its own coefficient
- missing categorical values were represented explicitly as a `"Missing"` category

This means that coefficients explain how the **fitted pipeline** uses the processed inputs, not how the original raw variables act in the world.

### What explainability means here

In this context, **explainability** means that we can inspect the fitted model directly and describe:

- which transformed features push predictions upwards or downwards
- which features have stronger or weaker effects in the model
- how the model is making its classification decisions overall

This is a form of **global model explanation**.

### Important considerations

These coefficients do **not** show causal effects.

They also do **not** automatically tell us that a feature is universally important outside the model.

Coefficient interpretation can be affected by:

- feature scaling
- feature correlation
- preprocessing choices
- the difference between conditional relationships and marginal relationships

So coefficients should be read as explanations of the **fitted model**, but cannot really make statements about cause and effect

## Inspection modes 

In [ ]:
# Start from the full coefficient table already created earlier
logistic_coefficient_summary_df = logistic_coefficients_df.copy()

# Add absolute coefficient magnitude and odds ratios
logistic_coefficient_summary_df["abs_coefficient"] = (
    logistic_coefficient_summary_df["coefficient"].abs()
)
logistic_coefficient_summary_df["odds_ratio"] = np.exp(
    logistic_coefficient_summary_df["coefficient"]
)

# Split the transformed feature name into parts
# Examples:
# num__capital_gain
# cat__marital_status_Married-civ-spouse
feature_parts = logistic_coefficient_summary_df["feature"].str.split("__", n=1, expand=True)
logistic_coefficient_summary_df["feature_type"] = feature_parts[0]
logistic_coefficient_summary_df["feature_name_processed"] = feature_parts[1]

# Create readable feature type labels
logistic_coefficient_summary_df["feature_type"] = (
    logistic_coefficient_summary_df["feature_type"]
    .replace({"num": "numerical", "cat": "categorical"})
)

# IMPORTANT:
# some original feature names contain underscores
# e.g. marital_status, education_num, native_country, hours_per_week
# for categorical variables, we need to match the full original feature name
# before separating the encoded category level

sorted_categorical_features = sorted(categorical_features, key=len, reverse=True)

def split_processed_feature(feature_name: str, feature_type: str):
    if feature_type == "numerical":
        return feature_name, np.nan

    for original_feature in sorted_categorical_features:
        prefix = f"{original_feature}_"
        if feature_name.startswith(prefix):
            level = feature_name[len(prefix):]
            return original_feature, level

    # Fallback if no categorical prefix matches
    return feature_name, np.nan

split_values = logistic_coefficient_summary_df.apply(
    lambda row: split_processed_feature(
        row["feature_name_processed"],
        row["feature_type"]
    ),
    axis=1
)

logistic_coefficient_summary_df["base_variable"] = [x[0] for x in split_values]
logistic_coefficient_summary_df["level"] = [x[1] for x in split_values]

# Direction of effect in the fitted model
logistic_coefficient_summary_df["direction"] = np.where(
    logistic_coefficient_summary_df["coefficient"] > 0,
    "towards >50K",
    np.where(
        logistic_coefficient_summary_df["coefficient"] < 0,
        "towards <=50K",
        "no directional effect"
    )
)

# Reorder columns for readability
logistic_coefficient_summary_df = logistic_coefficient_summary_df[
    [
        "feature",
        "feature_type",
        "base_variable",
        "level",
        "coefficient",
        "abs_coefficient",
        "odds_ratio",
        "direction",
    ]
]

# Save the full structured coefficient summary
coefficient_summary_file = PROCESSED_DIR / "01_logreg_04_coefficient_summary.csv"
logistic_coefficient_summary_df.to_csv(coefficient_summary_file, index=False)

print("\nStructured coefficient summary shape:")
print(logistic_coefficient_summary_df.shape)

print("\nFirst 10 rows:")
display(logistic_coefficient_summary_df.head(10))

# You will get a file save to inspect if you wish -> Outputs/Processed_data/logistic_regression_coefficient_summary.csv

In [ ]:
# Select the strongest positive and negative coefficients
top_positive_coefficients_df = logistic_coefficient_summary_df.nlargest(10, "coefficient").copy()
top_negative_coefficients_df = logistic_coefficient_summary_df.nsmallest(10, "coefficient").copy()

coefficient_plot_summary_df = pd.concat(
    [top_positive_coefficients_df, top_negative_coefficients_df],
    axis=0
).sort_values("coefficient")

# Create a signed coefficient plot
plt.figure(figsize=(10, 8))

bar_colors = [
    "#1f77b4" if value < 0 else "#d62728"
    for value in coefficient_plot_summary_df["coefficient"]
]

plt.barh(
    coefficient_plot_summary_df["feature"],
    coefficient_plot_summary_df["coefficient"],
    color=bar_colors
)

plt.axvline(x=0, color="black", linewidth=1)
plt.xlabel("Coefficient value")
plt.ylabel("Transformed feature")
plt.title("Top positive and negative Logistic Regression coefficients")
plt.tight_layout()

signed_coefficients_png = FIGURE_DIR / "01_logreg_05_coefficients.png"
signed_coefficients_svg = FIGURE_DIR / "01_logreg_05_coefficients.svg"

plt.savefig(signed_coefficients_png, dpi=300, bbox_inches="tight")
plt.savefig(signed_coefficients_svg, bbox_inches="tight")
plt.show()

# Save the subset used for plotting
coefficient_plot_subset_file = PROCESSED_DIR / "01_logreg_05_coefficients_subset.csv"
coefficient_plot_summary_df.to_csv(coefficient_plot_subset_file, index=False)

# Images also saved: logistic_regression_signed_coefficients in png and svg format

In [ ]:
# Select a small set of interpretable features for odds-ratio inspection
interpretable_features_for_or = [
    "num__capital_gain",
    "num__education_num",
    "num__age",
    "num__hours_per_week",
    "cat__marital_status_Married-civ-spouse",
    "cat__marital_status_Never-married",
    "cat__relationship_Wife",
    "cat__relationship_Own-child"
]

odds_ratio_plot_df = logistic_coefficient_summary_df[
    logistic_coefficient_summary_df["feature"].isin(interpretable_features_for_or)
].copy()

# Sort for plotting
odds_ratio_plot_df = odds_ratio_plot_df.sort_values("odds_ratio")

plt.figure(figsize=(10, 6))

or_colors = [
    "#1f77b4" if value < 1 else "#d62728"
    for value in odds_ratio_plot_df["odds_ratio"]
]

plt.barh(
    odds_ratio_plot_df["feature"],
    odds_ratio_plot_df["odds_ratio"],
    color=or_colors
)

plt.axvline(x=1, color="black", linewidth=1, linestyle="--")
plt.xlabel("Odds ratio = exp(coefficient)")
plt.ylabel("Transformed feature")
plt.title("Selected Logistic Regression odds ratios")
plt.tight_layout()

odds_ratio_png = FIGURE_DIR / "01_logreg_06_odds_ratios_selected.png"
odds_ratio_svg = FIGURE_DIR / "01_logreg_06_odds_ratios_selected.svg"

plt.savefig(odds_ratio_png, dpi=300, bbox_inches="tight")
plt.savefig(odds_ratio_svg, bbox_inches="tight")
plt.show()

# Save the odds-ratio subset used for plotting
odds_ratio_subset_file = PROCESSED_DIR / "01_logreg_06_odds_ratios_selected.csv"
odds_ratio_plot_df.to_csv(odds_ratio_subset_file, index=False)

### How to read the plot

The dashed vertical line at:

$$
\text{Odds Ratio} = 1
$$

marks the point of **no change in odds**.

- if the odds ratio is **greater than 1**, the feature increases the odds of the positive class in the fitted model
- if the odds ratio is **less than 1**, the feature decreases the odds of the positive class in the fitted model
- the further the bar is from 1, the stronger the effect in the fitted model

### What this means for the selected features

In this plot:

- `num__capital_gain` has the largest odds ratio, so it is one of the strongest positive signals pushing predictions towards `>50K`
- `cat__marital_status_Married-civ-spouse` and `cat__relationship_Wife` also increase the odds of `>50K`
- `num__education_num`, `num__hours_per_week`, and `num__age` have odds ratios above 1, so they also push the model towards `>50K`
- `cat__marital_status_Never-married` and `cat__relationship_Own-child` have odds ratios below 1, so they push the model towards `<=50K`

### Important interpretation note

These odds ratios explain how the **fitted Logistic Regression pipeline** uses the **processed feature space**.

What that means in terms of processed feature space is that we work with the output from our pre-processing choices

- numerical features were standardised before model fitting
- categorical features were one-hot encoded into indicator columns

So these odds ratios do **not** refer directly to the raw original variables in their original units.

For example:

- an odds ratio for a numerical feature refers to a **one standard deviation increase** in that standardised feature
- an odds ratio for a categorical feature refers to the presence of that encoded category indicator

### What this plot is useful for

This plot is useful because odds ratios are often easier to explain than raw coefficients:

- coefficients are expressed on the **log-odds** scale
- odds ratios convert those into a multiplicative change in odds

This gives a more intuitive explanation of direction and strength, while still keeping the connection to the underlying Logistic Regression model.

### NOTE

> These are still **model-based associations**, not causal effects.

They show how the fitted model is using the transformed features to separate the classes

## Reflection questions

1. Which three features have the clearest directional interpretation in the fitted Logistic Regression model?

2. Which coefficients are easy to explain in plain language, and which seem harder to justify or trust?

3. How did preprocessing change the meaning of the coefficients?

4. Why is it more accurate to say that the coefficients explain the **fitted model** rather than the real-world data-generating process?

5. Looking at the confusion matrix and the coefficient plot together, what can we say about both:
   - how well the model performs
   - how the model is making its decisions?

# Decision trees

rule-based interpretable mode
A Decision Tree makes predictions by learning a sequence of splits on the input features. Each path from the root of the tree to a leaf can be read as a decision rule.

In this notebook, we use a **shallow tree** so that the learned structure remains readable. This helps us compare two different kinds of interpretability:

- **Logistic Regression**, which explains predictions through coefficients
- **Decision Tree**, which explains predictions through rules and split conditions

We keep the same training and validation split so that model comparisons remain fair. CAUTION: this decision tree example uses a very simplistic rule heuristic. 

In [ ]:
# Tree-specific preprocessing:
# keep numerical features on their original scale
# and encode categoricals without scaling

tree_numerical_features = numerical_features
tree_categorical_features = categorical_features

tree_numerical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

tree_categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", tree_numerical_preprocessor, tree_numerical_features),
        ("cat", tree_categorical_preprocessor, tree_categorical_features)
    ]
)

decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=3,
            random_state=42
        ))
    ]
)

decision_tree_pipeline.fit(X_train, y_train)
y_valid_pred_decision_tree = decision_tree_pipeline.predict(X_valid)

print("Decision Tree validation accuracy:")
print(accuracy_score(y_valid, y_valid_pred_decision_tree))

print("\nDecision Tree validation classification report:")
print(classification_report(y_valid, y_valid_pred_decision_tree))

## Visualising Decision Tree performance and structure

We now inspect two things:

- the **confusion matrix**, to understand how the Decision Tree performs on the validation set
- the **tree structure itself**, to see the learned decision rules

This is one of the advantages of shallow trees in explainable machine learning: we can inspect both model performance and the learned structure directly.

In [ ]:
decision_tree_confusion_matrix_png = FIGURE_DIR / "02_tree_01_confusion_matrix.png"
decision_tree_confusion_matrix_svg = FIGURE_DIR / "02_tree_01_confusion_matrix.svg"


decision_tree_cm_fig, decision_tree_cm_ax = plt.subplots(figsize=(8, 5))

# Draw the confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_true=y_valid,
    y_pred=y_valid_pred_decision_tree,
    display_labels=decision_tree_pipeline.classes_,
    cmap="Greens",
    ax=decision_tree_cm_ax,
    colorbar=True
)


decision_tree_cm_ax.grid(False)
decision_tree_cm_ax.set_title("Confusion Matrix - Decision Tree")
decision_tree_cm_fig.tight_layout()

# Save the figure in svg png
decision_tree_cm_fig.savefig(
    decision_tree_confusion_matrix_png,
    dpi=300,
    bbox_inches="tight"
)
decision_tree_cm_fig.savefig(
    decision_tree_confusion_matrix_svg,
    bbox_inches="tight"
)

plt.show()

# will be saved in figures

In [ ]:
# Extract the transformed feature names used by the Decision Tree
decision_tree_feature_names_after_preprocessing = (
    decision_tree_pipeline.named_steps["preprocessor"].get_feature_names_out()
)

# Access the fitted tree model directly
fitted_decision_tree_model = decision_tree_pipeline.named_steps["model"]

# Plot and save the fitted Decision Tree
decision_tree_plot_png = FIGURE_DIR / "02_tree_02_structure.png"
decision_tree_plot_svg = FIGURE_DIR / "02_tree_02_structure.svg"

plt.figure(figsize=(20, 10))
plot_tree(
    fitted_decision_tree_model,
    feature_names=decision_tree_feature_names_after_preprocessing,
    class_names=decision_tree_pipeline.classes_,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Decision Tree Structure")
plt.tight_layout()
plt.savefig(decision_tree_plot_png, dpi=300, bbox_inches="tight")
plt.savefig(decision_tree_plot_svg, bbox_inches="tight")
plt.show()

## Interpreting the Decision Tree

A Decision Tree explains its predictions through a sequence of feature-based rules.

Each internal node asks a question of the form:

$$
x_j \le t
$$

where \(x_j\) is one transformed feature and \(t\) is a split threshold.

The tree chooses splits that reduce class impurity. In this notebook, impurity is measured using the **Gini impurity**:

$$
G = 1 - \sum_{k=1}^{K} p_k^2
$$

For binary classification, this becomes:

$$
G = 2p(1-p)
$$

where \(p\) is the proportion of one class in the node.

- lower Gini values mean the node is more pure
- higher Gini values mean the node contains a stronger mixture of classes

The tree can therefore be read as a set of if–then rules that partition the processed feature space into decision regions.

#### Extra details on the tree
### Split rule

At the top of the node, the tree shows a condition such as:

$$
x_j \le t
$$

where:

- \(x_j\) is one transformed feature
- \(t\) is the threshold used for the split

If the condition is **True**, follow the left branch.  
If the condition is **False**, follow the right branch.

### Gini impurity

The tree shows the **Gini impurity** for the node:

$$
G = 1 - \sum_{k=1}^{K} p_k^2
$$

For binary classification, this becomes:

$$
G = 2p(1-p)
$$

where \(p\) is the proportion of one class in the node.

- \(G = 0\) means the node is completely pure
- larger values mean the node contains a stronger mixture of classes

The algorithm chooses splits that reduce impurity.

###  Samples

This shows how many training examples reached that node.

### Value

This shows the class counts in the node, for example:

$$
[\text{count of } \leq 50K,\ \text{count of } >50K]
$$

This helps us see which class dominates the node.

### Class

This is the majority class prediction for that node.

### How to interpret a full path

A path from the root node to a leaf node can be read as a rule.

For example, a path like:

- `cat__marital_status_Married-civ-spouse <= 0.5`
- `num__capital_gain <= 0.824`
- `num__education_num <= 0.937`

means that if all three conditions are satisfied, the model predicts the class shown in the final leaf.

This tree highlights some of the same main signals seen earlier in Logistic Regression, including:

- marital status
- capital gain
- education level
- age

The two models express those signals differently:

- **Logistic Regression** uses coefficients
- **Decision Tree** uses threshold-based rules

Saving at interim

In [ ]:
# Save the train/validation split

X_train_file = PROCESSED_DIR / "X_train.pkl"
X_valid_file = PROCESSED_DIR / "X_valid.pkl"
y_train_file = PROCESSED_DIR / "y_train.pkl"
y_valid_file = PROCESSED_DIR / "y_valid.pkl"

X_train.to_pickle(X_train_file)
X_valid.to_pickle(X_valid_file)
y_train.to_pickle(y_train_file)
y_valid.to_pickle(y_valid_file)

# EBM Explainable Boosting Machine

Additive model **Explainable Boosting Machine (EBM)**. 

EBM is a glassbox additive model that is more flexible than Logistic Regression, but still remains interpretable. Instead of assigning a single coefficient to each feature, EBM learns a feature-specific contribution function.

The model can be written as:

$$
g(E[y]) = \beta_0 + \sum_j f_j(x_j)
$$

- \(g\) is the link function
- \(\beta_0\) is the intercept
- \(f_j(x_j)\) is the learned contribution function for feature \(x_j\)

This means that each feature contributes additively to the prediction, but the contribution does not need to be linear.

In [ ]:
# checkpoint
X_train_base = X_train.copy()
X_valid_base = X_valid.copy()
y_train_base = y_train.copy()
y_valid_base = y_valid.copy()

print("Base training shape:", X_train_base.shape)
print("Base validation shape:", X_valid_base.shape)
print("Columns used for EBM:", X_train_base.columns.tolist())

In [ ]:
# EBM uses minimally processed raw input:
# - no scaling
# - no one-hot encoding
# - explicit handling of missing values

ebm_numerical_features = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

ebm_categorical_features = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country"
]

# Copy raw inputs
X_train_ebm = X_train_base.copy()
X_valid_ebm = X_valid_base.copy()

# Ensure numeric columns are numeric
for frame in [X_train_ebm, X_valid_ebm]:
    frame[ebm_numerical_features] = frame[ebm_numerical_features].apply(
        pd.to_numeric,
        errors="coerce"
    )

# Simple, explicit imputation
numeric_fill_values = X_train_ebm[ebm_numerical_features].median()

X_train_ebm[ebm_numerical_features] = X_train_ebm[ebm_numerical_features].fillna(numeric_fill_values)
X_valid_ebm[ebm_numerical_features] = X_valid_ebm[ebm_numerical_features].fillna(numeric_fill_values)

X_train_ebm[ebm_categorical_features] = (
    X_train_ebm[ebm_categorical_features]
    .fillna("Missing")
    .astype(str)
)

X_valid_ebm[ebm_categorical_features] = (
    X_valid_ebm[ebm_categorical_features]
    .fillna("Missing")
    .astype(str)
)

# Explicit feature typing for EBM
ebm_feature_types = [
    "continuous" if column in ebm_numerical_features else "nominal"
    for column in X_train_ebm.columns
]

ebm_feature_metadata_df = pd.DataFrame({
    "feature": X_train_ebm.columns,
    "group": [
        "numerical" if column in ebm_numerical_features else "categorical"
        for column in X_train_ebm.columns
    ],
    "ebm_feature_type": ebm_feature_types
})

display(ebm_feature_metadata_df)
print("\nRemaining missing values in EBM training data:")
print(X_train_ebm.isna().sum())

# Model fit -> EBM

In [ ]:
ebm_classifier = ExplainableBoostingClassifier(
    feature_names=X_train_ebm.columns.tolist(),
    feature_types=ebm_feature_types,
    interactions=0,
    outer_bags=4,
    max_bins=128,
    max_rounds=2000,
    learning_rate=0.03,
    validation_size=0.15,
    early_stopping_rounds=50,
    n_jobs=1,
    random_state=42
)

ebm_fit_start = time.time()
ebm_classifier.fit(X_train_ebm, y_train_base)
ebm_fit_seconds = time.time() - ebm_fit_start

y_valid_pred_ebm = ebm_classifier.predict(X_valid_ebm)
y_valid_proba_ebm = ebm_classifier.predict_proba(X_valid_ebm)[:, 1]

print("EBM classes:", ebm_classifier.classes_)
#EBM can be employed comprehensively, here the parameters were reduced for brevity and time

In [ ]:
# Evaluate validation performance
print("EBM validation accuracy:")
print(accuracy_score(y_valid_base, y_valid_pred_ebm))

print("\nEBM validation classification report:")
print(classification_report(y_valid_base, y_valid_pred_ebm))

# Save classification report
ebm_classification_report_df = pd.DataFrame(
    classification_report(
        y_valid_base,
        y_valid_pred_ebm,
        output_dict=True
    )
).T

ebm_classification_report_file = PROCESSED_DIR / "03_ebm_01_classification_report.csv"
ebm_classification_report_df.to_csv(ebm_classification_report_file)

# Save validation predictions
ebm_validation_predictions_df = X_valid_ebm.copy()
ebm_validation_predictions_df["y_true"] = y_valid_base.values
ebm_validation_predictions_df["y_pred"] = y_valid_pred_ebm
ebm_validation_predictions_df["probability_>50K"] = y_valid_proba_ebm

ebm_validation_predictions_file = PROCESSED_DIR / "03_ebm_02_validation_predictions.csv"
ebm_validation_predictions_df.to_csv(ebm_validation_predictions_file, index=True)

In [ ]:
plt.rcParams["axes.grid"] = False

ebm_confusion_matrix_png = FIGURE_DIR / "03_ebm_03_confusion_matrix.png"
ebm_confusion_matrix_svg = FIGURE_DIR / "03_ebm_03_confusion_matrix.svg"

ebm_cm_fig, ebm_cm_ax = plt.subplots(figsize=(8, 5))

ConfusionMatrixDisplay.from_predictions(
    y_true=y_valid_base,
    y_pred=y_valid_pred_ebm,
    display_labels=ebm_classifier.classes_,
    cmap="Purples",
    ax=ebm_cm_ax,
    colorbar=True
)

ebm_cm_ax.grid(False)
ebm_cm_ax.set_title("Confusion Matrix - Explainable Boosting Machine")
ebm_cm_fig.tight_layout()

ebm_cm_fig.savefig(ebm_confusion_matrix_png, dpi=300, bbox_inches="tight")
ebm_cm_fig.savefig(ebm_confusion_matrix_svg, bbox_inches="tight")

plt.show()

In [ ]:
set_visualize_provider(InlineProvider())
pio.renderers.default = "plotly_mimetype"

# Global explanation
ebm_global = ebm_classifier.explain_global(name="EBM global explanation")

# Overall term importances
ebm_term_importance_df = pd.DataFrame({
    "term": ebm_classifier.term_names_,
    "importance": ebm_classifier.term_importances()
}).sort_values("importance", ascending=False)

ebm_term_importance_file = PROCESSED_DIR / "03_ebm_04_term_importances.csv"
ebm_term_importance_df.to_csv(ebm_term_importance_file, index=False)

display(ebm_term_importance_df.head(10))

# Show the explanation inline in the notebook
show(ebm_global)

# Save the overall global explanation
ebm_global_overall_html = FIGURE_DIR / "03_ebm_05_global_overall.html"
preserve(ebm_global, file_name=str(ebm_global_overall_html))

### Reading this plot

- **Score** = the contribution of this feature to the model output
- **Positive score** = pushes the model more toward the positive class
- **Negative score** = pushes the model away from the positive class
- **Density** = where the data are concentrated for this feature
- Interpret patterns in **dense regions** with more confidence than patterns in sparse regions

For classification, the score is usually shown on the **logit** scale rather than as a probability.

Each EBM plot shows the learned contribution of one feature or term to the model prediction.

For a **continuous feature** such as `age` or `hours_per_week`:

- the **x-axis** shows the feature values
- the **y-axis** shows the feature contribution to the model score

For a **categorical feature** such as `marital_status`:

- the **x-axis** shows the categories
- the **y-axis** shows the contribution associated with each category

### What does "score" mean?

The **score** is the contribution of that term to the model output before converting it into a final class probability.

For binary classification, the EBM is additive on the **logit** scale:

$$
\text{score}(x) = \beta_0 + \sum_j f_j(x_j)
$$

- $\beta_0$ is the intercept
- $f_j(x_j)$ is the contribution from feature $j$

A **positive score contribution** pushes the prediction more toward the positive class.

A **negative score contribution** pushes the prediction away from the positive class.

So the plot does **not** show the raw probability directly.  
It shows how much that feature raises or lowers the model score before the logistic transformation is applied.

The **density** at the bottom of the plot is a histogram showing where the data are located for that feature.

This is useful because a feature effect is usually more trustworthy where the model saw plenty of training data, and less stable where data are sparse.

For example:

- if a region has **high density**, many observations were available there
- if a region has **low density**, few observations were available there

- Where is the model seeing many training examples?
- Where is the curve based on sparse data and therefore more uncertain?

### How should we interpret a plot?

1. Is the feature effect roughly linear or clearly non-linear?
2. Are there thresholds, plateaus, or sharp jumps?
3. Where does the feature push predictions upward or downward?
4. Is the pattern occurring in a dense part of the data, or only in a sparse tail?

This is what makes EBM useful for explainability:
it keeps the model interpretable, while allowing more flexible feature-response shapes than a single linear coefficient like the log regression

Important:
the score in a feature plot is **not** the final prediction by itself.
It is only one additive part of the final model output.

In [ ]:
# Display and save specific EBM term plots
top_terms_to_plot = [
    ("age", "03_ebm_06_term_plot_age.html"),
    ("marital_status", "03_ebm_07_term_plot_marital_status.html"),
    ("capital_gain", "03_ebm_08_term_plot_capital_gain.html"),
    ("hours_per_week", "03_ebm_09_term_plot_hours_per_week.html"),
]

for term_name, output_name in top_terms_to_plot:
    term_index = list(ebm_classifier.term_names_).index(term_name)
    fig = ebm_global.visualize(term_index)
    fig.show()

    html_path = FIGURE_DIR / output_name
    fig.write_html(
        str(html_path),
        include_plotlyjs=True,
        full_html=True
    )

## Local EBM explanations

A local EBM explanation shows how one prediction is built from the intercept and the individual term contributions.

For one case, the model score is:

$$
\text{score}(x) = \beta_0 + \sum_j f_j(x_j)
$$

where:

- $\beta_0$ is the intercept
- $f_j(x_j)$ is the contribution from feature $j$

This means the prediction is built by adding together the contribution from each feature, then converting the total score into a probability.

A local explanation helps answer:

- Which terms pushed this prediction upward?
- Which terms pushed it downward?
- Which terms mattered most for this one case?

The global plots describe the model overall. The local explanation shows how the model behaved for one individual prediction.

In [ ]:
# Explain a few validation cases
ebm_local = ebm_classifier.explain_local(X_valid_ebm.iloc[:5], y_valid_base.iloc[:5], name="EBM local explanations")

# Show the first local explanation
show(ebm_local, 0)

# Guided comparison of the three interpretable models

We have now fitted three interpretable models on the same task:

- **Logistic Regression**
- **Decision Tree**
- **Explainable Boosting Machine (EBM)**

The aim is to compare predictive performance and explanation format.

1. How well did the model perform on the validation data?
2. What kind of explanation does the model give?
3. When is that explanation form most useful, and what is its main limitation?

Different interpretable models support different kinds of understanding:

- Logistic Regression explains through **coefficients**
- Decision Trees explain through **rules and thresholds**
- EBMs explain through **feature-shape functions**

A more accurate model is not always the easiest model to explain, and the easiest model to explain is not always the most faithful summary of the relationship in the data.